# Aggregating 1 Minute OHLCV bars into 5 minutes bars using Nautilus Trader's in-built "Bar Aggregators"

In [22]:
import pandas as pd
from nautilus_trader.persistence.catalog import ParquetDataCatalog 
from nautilus_trader.model.data import (
    Bar, BarType , BarAggregation, BarSpecification)
from nautilus_trader.model.enums import PriceType
from nautilus_trader.data.aggregation import TimeBarAggregator

from nautilus_trader.model.identifiers import InstrumentId
from nautilus_trader.model.objects import Price, Quantity
from decimal import Decimal
from nautilus_trader.core.datetime import millis_to_nanos

CSV_PATH = "./data/sample_1min_ohlcv_2025_unix_timestamp.csv"
CATALOG_PATH = "./catalog" 
INSTRUMENT_ID = InstrumentId.from_str("BTCUSDT.BINANCE")

df = pd.read_csv(CSV_PATH)
catalog = ParquetDataCatalog(CATALOG_PATH)
df.head()

,timestamp,open,high,low,close,volume
0,1735722900000,25000.00,25003.93,24999.84,25001.24,1288
1,1735722960000,25001.24,25001.82,24999.72,25000.90,5247
2,1735723020000,25000.90,25003.05,24998.63,25002.52,3543
3,1735723080000,25002.52,25008.10,25001.91,25006.32,5784
4,1735723140000,25006.32,25008.33,25003.88,25005.74,1421


# Creating Bar Type

In [23]:
bar_spec = BarSpecification(
    step=1,
    aggregation=BarAggregation.MINUTE,
    price_type=PriceType.LAST,
)

bar_type = BarType(
    instrument_id = INSTRUMENT_ID,
    bar_spec = bar_spec,
)



In [24]:
bars = []

for _, row in df.iterrows():

    ts_event = millis_to_nanos(int(row["timestamp"]))

    bar = Bar(
    bar_type=bar_type,

    open=Price.from_str(str(row["open"])),
    high=Price.from_str(str(row["high"])),
    low=Price.from_str(str(row["low"])),
    close=Price.from_str(str(row["close"])),

    volume=Quantity.from_int(int(row["volume"])),

    ts_event=ts_event,
    ts_init=ts_event,
    )

    bars.append(bar)

print(f"Created {len(bars)} bars","\n")

Created 8648 bars 



# Writing the bars to ParquetDataCatalog

In [25]:
catalog = ParquetDataCatalog(CATALOG_PATH)
catalog.write_data(bars)
print("Written to ParquetCatalog")

File c:/Users/user/OneDrive/Desktop/nautilus_trader_practice/catalog/data/bar/BTCUSDT.BINANCE-1-MINUTE-LAST-EXTERNAL/2025-01-01T09-15-00-000000000Z_2025-01-31T15-30-00-000000000Z.parquet already exists, skipping write
Written to ParquetCatalog


# Aggregation Pipeline
- Now we will work with the earlier created ParquetDataCatalog.
- We will try to generate the following timestamps:
- 5 Min, 30 Min, 1 Hour, 2 Hour, 1 day, 1 Week, 1 Month

In [26]:
catalog = ParquetDataCatalog(CATALOG_PATH)

bar_spec_5m = BarSpecification(
    step = 5,
    aggregation = BarAggregation.MINUTE,
    price_type = PriceType.MID
)

bar_type_5m = BarType(
    bar_spec = bar_spec_5m,
    instrument_id = INSTRUMENT_ID
)

aggregator = TimeBarAggregator(
    bar_type = bar_type_5m 
)

# Nautilus Aggregator acts as even-driven (converts each bar in a stream of bars to specifiec bar_type) 
aggregated_bars = []

for bar in bars:
    result = aggregator.handle_bar(bar)

    if(result is not None):
        aggregated_bars.append(result)


print(f"Created{len(aggregated_bars)} aggregated bars")


TypeError: __init__() takes at least 4 positional arguments (0 given)